# Delta Lake — Full Deep Dive

## What Delta Lake Adds on Top of Parquet

Plain Parquet is an excellent columnar file format, but a *directory of Parquet files* is not a *database table*. It lacks:

| Capability | Plain Parquet | Delta Lake |
|---|---|---|
| **ACID transactions** | No — concurrent writers corrupt data | Yes — serialisable isolation via an OCC log |
| **DML (UPDATE / DELETE / MERGE)** | No — files are immutable; you must rewrite | Yes — row-level DML via copy-on-write or merge-on-read |
| **Time travel** | No — once rewritten, old data is gone | Yes — every version is addressable by version number or timestamp |
| **Schema enforcement** | No — a writer can add/drop/rename columns silently | Yes — writes that violate the schema are rejected at the source |
| **Schema evolution** | Manual partition repair | Controlled via `mergeSchema` / `overwriteSchema` options |
| **Audit history** | None | Full commit history in `_delta_log` |

## The Transaction Log — How ACID Actually Works

Delta Lake's secret ingredient is the **`_delta_log` directory** that lives alongside the Parquet data files. It is an **append-only log of JSON commit files**, one per transaction:

```
/home/jovyan/data/delta/orders/
├── _delta_log/
│   ├── 00000000000000000000.json   ← version 0: initial write
│   ├── 00000000000000000001.json   ← version 1: first UPDATE
│   ├── 00000000000000000002.json   ← version 2: first DELETE
│   ├── 00000000000000000003.json   ← version 3: MERGE
│   └── 00000000000000000010.checkpoint.parquet  ← checkpoint every 10 versions
├── part-00000-....snappy.parquet   ← data files (never mutated, only added/removed)
├── part-00001-....snappy.parquet
└── ...
```

Each JSON commit file contains **actions**:
- `add`: a new Parquet file is part of the table.
- `remove`: a Parquet file is no longer part of the table (soft delete — file still on disk until VACUUM).
- `metaData`: schema changes.
- `commitInfo`: who committed, when, what operation.

**How ACID is achieved**:
- **Atomicity**: a transaction either appends a new JSON commit file (success) or does not (failure). There is no partial state.
- **Consistency**: schema enforcement rules are checked before the commit file is written.
- **Isolation**: Delta uses **Optimistic Concurrency Control (OCC)**. Writers proceed independently, then at commit time check whether any conflicting transaction was committed between when they started and when they try to commit. If yes, the write is retried or rejected. Readers get **snapshot isolation** — they always see a consistent snapshot corresponding to a single version, even if another writer is mid-transaction.
- **Durability**: the log files are stored on HDFS / S3 / GCS / ADLS, which provide their own durability guarantees.

**Snapshot isolation in plain English**: when you `spark.read.format('delta').load(path)`, Delta finds the latest version in `_delta_log`, reconstructs the complete file list for that version (union of all `add` actions minus all `remove` actions up to that version), and reads exactly those files. You never see a partially-committed write from another concurrent transaction.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType, IntegerType
from delta.tables import DeltaTable
import os

# Delta Lake requires two configuration items:
#   1. spark.sql.extensions — loads the Delta SQL parser extensions
#      (enables DESCRIBE HISTORY, VACUUM, RESTORE, etc. as SQL commands)
#   2. spark.sql.catalog.spark_catalog — replaces the default Hive catalog
#      so that `spark.table("name")` can resolve Delta tables by path or name

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("DeltaLake_Full")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog"
    )
    # Let AQE handle partition coalescing
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.shuffle.partitions", "50")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

DATA_DIR   = "/home/jovyan/data"
DELTA_PATH = f"{DATA_DIR}/delta/orders"

print(f"Spark {spark.version} with Delta Lake ready")
print(f"Delta table path: {DELTA_PATH}")

In [ ]:
# ── Step 1: Write initial DataFrame as a Delta table ─────────────────────────
# This creates version 0 in _delta_log.
# The write is atomic: either ALL Parquet files are committed (and the
# version-0 JSON file is written), or none are visible to readers.

# Define a deterministic schema so schema enforcement can do its job later
orders_schema = StructType([
    StructField("order_id",    LongType(),   nullable=False),
    StructField("customer_id", IntegerType(),nullable=False),
    StructField("status",      StringType(), nullable=True),
    StructField("amount",      DoubleType(), nullable=True),
    StructField("region",      StringType(), nullable=True),
])

orders_df = spark.createDataFrame(
    [
        (1, 101, "pending",   150.0, "US"),
        (2, 102, "shipped",   320.5, "EU"),
        (3, 103, "pending",    85.0, "US"),
        (4, 104, "delivered", 210.0, "APAC"),
        (5, 105, "pending",   560.0, "EU"),
        (6, 106, "shipped",    40.0, "US"),
        (7, 107, "cancelled",  95.0, "APAC"),
        (8, 108, "pending",   730.0, "US"),
    ],
    schema=orders_schema
)

(
    orders_df
    .write
    .format("delta")          # use Delta instead of parquet/json/csv
    .mode("overwrite")        # overwrite if table already exists (creates v0)
    .save(DELTA_PATH)
)

print(f"Delta table written to: {DELTA_PATH}")
print(f"Row count: {spark.read.format('delta').load(DELTA_PATH).count()}")

In [ ]:
# ── Step 2: Read the Delta table and inspect the transaction log ──────────────
# Delta materialises the table state by replaying the _delta_log from the
# beginning (or the latest checkpoint) up to the requested version.

orders = spark.read.format("delta").load(DELTA_PATH)
print("Current table contents (version 0):")
orders.show()
print(f"Schema: {orders.schema.simpleString()}")
print()

# Inspect the transaction log directly — the _delta_log is just files
log_df = spark.read.json(f"{DELTA_PATH}/_delta_log/")
print("Transaction log structure (first commit file):")
# Each commit JSON has top-level keys: add, remove, metaData, commitInfo, protocol
log_df.select("commitInfo.operation", "commitInfo.timestamp",
              "add.path", "add.size").show(20, truncate=60)

# Use Delta's high-level history API (much more readable)
delta_table = DeltaTable.forPath(spark, DELTA_PATH)
print("DESCRIBE HISTORY:")
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=50)

In [ ]:
# ── Step 3: DML — INSERT, UPDATE, DELETE ─────────────────────────────────────
# Unlike plain Parquet, Delta supports row-level mutations.
# Under the hood, Delta uses "copy-on-write" (CoW) by default:
#   - Identify which Parquet files contain the rows to change.
#   - Read those files, apply the mutation in memory.
#   - Write NEW Parquet files with the updated rows.
#   - Commit a transaction that adds the new files and removes the old ones.
# The old files are NOT deleted — they are just marked as removed in _delta_log,
# which is what enables time travel.  VACUUM eventually cleans them up.

# ── INSERT (append new rows) ──────────────────────────────────────────────────
new_orders = spark.createDataFrame(
    [
        (9,  109, "pending", 420.0, "EU"),
        (10, 110, "shipped", 180.0, "US"),
    ],
    schema=orders_schema
)

# mode("append") creates a new commit in _delta_log (version 1)
new_orders.write.format("delta").mode("append").save(DELTA_PATH)
print(f"After INSERT: {spark.read.format('delta').load(DELTA_PATH).count()} rows")

# ── UPDATE ────────────────────────────────────────────────────────────────────
# Mark all "pending" orders in the US as "processing"
# This creates version 2
delta_table = DeltaTable.forPath(spark, DELTA_PATH)  # re-fetch after insert

delta_table.update(
    condition = F.expr("status = 'pending' AND region = 'US'"),
    set       = {"status": F.lit("processing")}  # set status column to the literal string
)
print("After UPDATE — US pending → processing:")
spark.read.format("delta").load(DELTA_PATH).filter("region = 'US'").show()

# ── DELETE ────────────────────────────────────────────────────────────────────
# Remove all cancelled orders.  Creates version 3.
delta_table.delete(condition = F.expr("status = 'cancelled'"))
print(f"After DELETE (cancelled): {spark.read.format('delta').load(DELTA_PATH).count()} rows")

In [ ]:
# ── Step 4: MERGE (upsert) ───────────────────────────────────────────────────
# MERGE is the most powerful DML operation.  It is equivalent to SQL MERGE:
#   - For rows that MATCH a condition: UPDATE or DELETE
#   - For rows that do NOT match: INSERT
# This is the canonical pattern for Change Data Capture (CDC) pipelines where
# you receive a stream of updates and want to apply them to a target table.

# Source DataFrame: a batch of changes arriving from an upstream system
# order_id 2: status changed to "delivered"
# order_id 5: amount changed
# order_id 99: brand-new order (not in the target yet → INSERT)
updates_df = spark.createDataFrame(
    [
        (2,  102, "delivered", 320.5, "EU"),    # existing → update status
        (5,  105, "shipped",   575.0, "EU"),    # existing → update amount
        (99, 199, "pending",   999.0, "US"),    # new → insert
    ],
    schema=orders_schema
)

delta_table = DeltaTable.forPath(spark, DELTA_PATH)

(
    delta_table.alias("target")
    .merge(
        source    = updates_df.alias("source"),
        condition = "target.order_id = source.order_id"  # join predicate
    )
    # When the join finds a match: overwrite all columns from source
    .whenMatchedUpdateAll()
    # When the join finds no match in target: insert the source row
    .whenNotMatchedInsertAll()
    .execute()  # creates a new commit in _delta_log
)

print("After MERGE:")
spark.read.format("delta").load(DELTA_PATH).orderBy("order_id").show()

# Show the updated history
delta_table.history().select("version", "timestamp", "operation").show()

In [ ]:
# ── Step 5: Time Travel ───────────────────────────────────────────────────────
# Because Delta never mutates Parquet files (it only adds new files and marks
# old ones as removed), every historical version of the table is still on disk.
# Time travel is simply a matter of telling Delta to replay the log only up to
# a specific version number or timestamp.
#
# Use cases:
#   - "Oops, I deleted the wrong rows" → read v(N-1), see what was there
#   - Reproducible ML experiments → pin training data to a specific version
#   - Auditing → who changed what and when

# ── Read at version 0 (the initial write, before any DML) ────────────────────
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)   # can also use .option("timestampAsOf", "2024-01-01")
    .load(DELTA_PATH)
)
print("Table at VERSION 0 (initial write):")
df_v0.show()
print(f"  Row count at v0: {df_v0.count()}")

# ── Read at version 2 (after UPDATE, before DELETE and MERGE) ────────────────
df_v2 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 2)
    .load(DELTA_PATH)
)
print("Table at VERSION 2 (after INSERT + UPDATE):")
df_v2.show()
print(f"  Row count at v2: {df_v2.count()}")

# ── Compare row counts across versions ───────────────────────────────────────
current_count = spark.read.format("delta").load(DELTA_PATH).count()
print()
print(f"  v0 (initial write)          : {df_v0.count()} rows")
print(f"  v2 (after insert+update)    : {df_v2.count()} rows")
print(f"  current (after all DML)     : {current_count} rows")

In [ ]:
# ── Step 6: Schema Evolution ──────────────────────────────────────────────────
# By default, Delta enforces schema strictly: writing a DataFrame whose schema
# does not match the table's schema raises an AnalysisException.
#
# To ADD new columns, use the .option("mergeSchema", "true") write option.
# This updates the metaData action in _delta_log with the new schema definition.
# Existing rows will have NULL for the new column.
#
# IMPORTANT: you cannot use mergeSchema to:
#   - Rename a column (Delta sees it as drop + add)
#   - Change a column's data type (e.g. IntegerType → LongType)
#   - Drop a column
# For those changes, use overwriteSchema=true (destructive — rewrites all data)
# or use ALTER TABLE SQL commands with column mapping enabled.

# Source DataFrame with an additional column: "priority"
evolved_orders = spark.createDataFrame(
    [
        (11, 111, "pending", 250.0, "US",   "high"),
        (12, 112, "shipped",  90.0, "APAC", "low"),
    ],
    schema=StructType([
        StructField("order_id",    LongType(),   nullable=False),
        StructField("customer_id", IntegerType(),nullable=False),
        StructField("status",      StringType(), nullable=True),
        StructField("amount",      DoubleType(), nullable=True),
        StructField("region",      StringType(), nullable=True),
        StructField("priority",    StringType(), nullable=True),   # NEW column
    ])
)

# Without mergeSchema=true this would fail with:
#   AnalysisException: A schema mismatch detected when writing to the Delta table
(
    evolved_orders
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")  # tell Delta to update schema definition
    .save(DELTA_PATH)
)

print("After schema evolution (mergeSchema=true):")
latest = spark.read.format("delta").load(DELTA_PATH)
print(f"New schema: {latest.schema.simpleString()}")
print()
# Older rows have NULL for 'priority' — schema evolution is non-destructive
latest.orderBy("order_id").select("order_id", "status", "priority").show()

In [ ]:
# ── Step 7: VACUUM — Cleaning Up Old Data Files ───────────────────────────────
# After many UPDATE / DELETE / MERGE operations, the table directory accumulates
# many "removed" Parquet files.  They are still referenced by old _delta_log
# entries and are needed for time travel within the retention window.
#
# VACUUM permanently deletes Parquet files that:
#   1. Are no longer part of the CURRENT table (i.e. marked as "remove" in the log), AND
#   2. Are older than the retention threshold (default: 7 days = 168 hours)
#
# IMPORTANT SAFETY RULES:
#   - NEVER set retention < 7 days in production unless you are certain no reader
#     or running job is using an older version.
#   - VACUUM 0 HOURS in production WILL break any in-flight time travel queries.
#   - After VACUUM, time travel to vacuumed versions is impossible.
#
# The DRY RUN option lists files that WOULD be deleted without actually deleting them.
# Always run DRY RUN first in production to audit the impact.

# Disable the safety check that prevents VACUUM with retention < 7 days.
# We do this ONLY for the educational dry run below.  Do NOT do this in production.
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

print("VACUUM DRY RUN — files that would be deleted (retention = 0 hours):")
print("(In production, use RETAIN 168 HOURS or higher)")
print()

# DRY RUN: list candidate files without deletion
vacuum_result = spark.sql(f"""
    VACUUM delta.`{DELTA_PATH}`
    RETAIN 0 HOURS
    DRY RUN
""")
vacuum_result.show(50, truncate=False)

print(f"Files that would be deleted: {vacuum_result.count()}")
print()
print("To actually vacuum in production:")
print(f"  VACUUM delta.`{DELTA_PATH}` RETAIN 168 HOURS")
print()
print("After vacuuming, time travel to deleted versions raises:")
print("  AnalysisException: No reproducible checkpoint found for version X")

# Re-enable the safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

In [ ]:
# ── Final summary: full history ───────────────────────────────────────────────
delta_table = DeltaTable.forPath(spark, DELTA_PATH)
print("Complete transaction history for the orders Delta table:")
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).orderBy("version").show(20, truncate=60)

# Stop the SparkSession
spark.stop()
print("SparkSession stopped. Delta Lake notebook complete.")